<a href="https://colab.research.google.com/github/xzhen-hub/ML_Linear-Factor-Model/blob/main/g1_julsep_saved_models_20261101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Matched July–September comparison: same saved Fannie models
Prepared November 1, 2026. This is a continuation of the completed 53-calendar-month comparison, with a NEW run directory. It does not change the old notebook, old ZIPs, either tree, or their training data.

**Primary:** compare July, August, September 2026 with exactly the same before-calibration prediction treatment as the completed experiment. **Separate sensitivity:** fit one June-only odds adjustment per arm over the same overall G1 population; freeze the factors before evaluating later months. A shared production-derived June factor is a diagnostic third stage. Neither sensitivity is claimed to reproduce deployed production calibration or an existing candidate calibration.

**Scope:** Fannie only; there is no saved expanded Freddie model in these inputs. Keep the original corrected verified pool roster, coupon buckets and adjusted vintage definition. Reconstruct each month's membership from servicing, without requiring survival into September or a previous-month row. Null curtailment training eligibility is diagnostic, not a missing payoff outcome.

**Storage:** the only persistent BigQuery write is a unique panel in `gen-lang-client-0343560978.oasive_quant_sandbox`, expiring after 30 days. Source/model reads are read-only. Local results, predictions and saved model are backed up in ZIPs. No bucket or additional production permissions.

**Interpretation:** later-calendar assessment, not a pristine blind test or a split-only causal experiment: July was already inspected during research. Both arms use the same historical feature snapshot; this does not certify historical point-in-time feature availability. September remains provisional until outcomes are complete.

## Files to upload
Open this notebook with Colab **File → Upload notebook**, save a copy in Google Drive, and use a CPU runtime. Through the Files sidebar upload just:

`g1_fannie_full53_colab_20261101_20261101T223733_7e881b55_FULL_recovery.zip`

It is saved at `Y:/OASIVE/`. Wait for the approximately 629 MB upload to finish. The small-results ZIP is insufficient. No older experiment ZIPs or old BigQuery panel are needed. Helpers are embedded/restored.

For runtime recovery, upload this notebook's own newest FULL output and set `RESTORE_FOLLOWUP_ZIP` in Step 1. Keep the root and code unchanged when resuming.


## Step 1 — Restore the saved models and references
Run after upload finishes. Restore selected verified files, not training data. If the upload has a different suffix, edit `INPUT_FULL` to the exact Colab path. No training or `extractall` is used.


In [3]:
from pathlib import Path
import zipfile

candidates = sorted(Path("/content").glob("*FULL_recovery*.zip"))
for p in candidates:
    print(p, "| bytes:", p.stat().st_size,
          "| readable ZIP:", zipfile.is_zipfile(p))

/content/g1_fannie_full53_colab_20261101_20261101T223733_7e881b55_FULL_recovery.zip | bytes: 629460318 | readable ZIP: True


In [4]:
from pathlib import Path
import json,zipfile,hashlib,shutil,sys,subprocess,importlib,importlib.util,gc,uuid
ROOT=Path('/content/g1_julsep_saved_models_20261101');ROOT.mkdir(exist_ok=True)
INPUT_FULL=Path('/content/g1_fannie_full53_colab_20261101_20261101T223733_7e881b55_FULL_recovery.zip')
RESTORE_FOLLOWUP_ZIP=None  # Resume only: '/content/<this-notebooks-newest-FULL_recovery.zip>'
INPUTS=ROOT/'inputs'
def file_sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
def restore_members(archive,destination,manifest_name,selected=None):
    archive=Path(archive);destination=Path(destination)
    assert archive.is_file() and zipfile.is_zipfile(archive),f'Missing/incomplete ZIP: {archive}'
    with zipfile.ZipFile(archive) as z:
        hashes=json.loads(z.read(manifest_name));names=list(hashes) if selected is None else selected
        for name in names:
            assert name in hashes,'Wrong ZIP; missing '+name
            dest=(destination/name).resolve()
            assert dest.is_relative_to(destination.resolve()),'Unsafe archive path'
            if not dest.exists():
                dest.parent.mkdir(parents=True,exist_ok=True);temp=dest.with_suffix(dest.suffix+'.recovering')
                with z.open(name) as src,temp.open('wb') as dst:shutil.copyfileobj(src,dst,1024*1024)
                assert file_sha(temp)==hashes[name],'Recovery checksum mismatch: '+name
                temp.replace(dest)
            assert file_sha(dest)==hashes[name],'Existing file conflicts with backup: '+name
        return {n:hashes[n] for n in names}
selected=['run_spec.json','preparation_spec.json','completion.json','march_bridge.json','pool_roster_audit.csv',
          'code/g1_saved_payoff_comparison_core.py','code/g1_fannie_full53.py',
          'recovered/expanded/manifest.json','recovered/expanded/experiment_spec.json',
          'recovered/expanded/post_rally_matched_test.py','recovered/expanded/expanded.ubj',
          'recovered/expanded/encoder.joblib','recovered/production/march_golden.parquet']
if RESTORE_FOLLOWUP_ZIP:
    restore_members(RESTORE_FOLLOWUP_ZIP,ROOT,'julsep_recovery_hashes.json')
elif not (INPUTS/'selected_input_hashes.json').exists():
    hashes=restore_members(INPUT_FULL,INPUTS,'monthly_recovery_hashes.json',selected)
    (INPUTS/'selected_input_hashes.json').write_text(json.dumps(hashes,indent=2))
for name,digest in json.loads((INPUTS/'selected_input_hashes.json').read_text()).items():
    assert file_sha(INPUTS/name)==digest,'Changed restored input: '+name
MANIFEST=json.loads((INPUTS/'recovered/expanded/manifest.json').read_text())
OLD_SPEC=json.loads((INPUTS/'run_spec.json').read_text())
assert OLD_SPEC['months'][0]=='2022-01-01' and OLD_SPEC['months'][-1]=='2026-05-01'
print('Saved inputs restored:',INPUTS)


Saved inputs restored: /content/g1_julsep_saved_models_20261101/inputs


## Step 2 — Match the saved package versions
If packages install, select **Runtime → Restart session**, then rerun Steps 1–2. Do not retrain for dependency problems. If a pinned version is unavailable for the runtime Python version, stop; do not silently substitute an encoder implementation.


In [5]:
import importlib.metadata
requirements=[]
for name,version in MANIFEST['versions'].items():
    try:current=importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:current=None
    if current!=version:requirements.append(name+'=='+version)
if requirements:
    subprocess.run([sys.executable,'-m','pip','install','--quiet',*requirements],check=True)
    raise RuntimeError('Packages restored. Restart session, rerun Steps 1–2, then continue.')
print('Exact recorded package versions match.')


Exact recorded package versions match.


## Step 3 — Write the new comparison helper
Run per fresh session. This is separate from the archived full53 helper. Do not paste the old repair cells into this notebook.


In [6]:
%%writefile /content/g1_julsep_saved_models_20261101.py
"""Matched later-calendar Fannie G1 evaluation; saved trees only; optional June sensitivity."""
from pathlib import Path
import json, hashlib, re, shutil, zipfile
import numpy as np
import pandas as pd
import g1_saved_payoff_comparison_core as core
import g1_fannie_full53 as old
PROJECT=core.PROJECT
FEATURES=old.FEATURES;LABELS=old.LABELS;SERVICING=old.SERVICING;POOLS=old.POOLS
MODEL=core.MODEL
CAL_MONTH='2026-06-01'
EVAL_MONTHS=['2026-07-01','2026-08-01','2026-09-01']
MONTHS=[CAL_MONTH]+EVAL_MONTHS
SOURCES=[FEATURES,LABELS,SERVICING,POOLS]
QUERY_CAP=500*1024**3
REPORT_CAP=10*1024**3
PLAN_CAP=750*1024**3
STAGES=['before','june_separate','june_common_production']

def wjson(p,d):core.write_json(p,d)
def stable(d):return hashlib.sha256(json.dumps(d,sort_keys=True,default=str).encode()).hexdigest()

def restore_selected(archive,root):
 root=Path(root);archive=Path(archive)
 if not zipfile.is_zipfile(archive):raise ValueError('Upload the completed full53 FULL_recovery ZIP; wait for upload completion')
 required=['run_spec.json','preparation_spec.json','completion.json','march_bridge.json','pool_roster_audit.csv',
 'code/g1_saved_payoff_comparison_core.py','code/g1_fannie_full53.py',
 'recovered/expanded/manifest.json','recovered/expanded/experiment_spec.json',
 'recovered/expanded/post_rally_matched_test.py','recovered/expanded/expanded.ubj',
 'recovered/expanded/encoder.joblib','recovered/production/march_golden.parquet']
 with zipfile.ZipFile(archive) as z:
  manifest=json.loads(z.read('monthly_recovery_hashes.json'))
  for name in required:
   if name not in manifest:raise ValueError('Wrong or small ZIP; missing '+name)
   dest=root/name;dest.parent.mkdir(parents=True,exist_ok=True)
   if not dest.exists():
    temp=dest.with_suffix(dest.suffix+'.recovering')
    with z.open(name) as src,temp.open('wb') as dst:shutil.copyfileobj(src,dst,1024**2)
    if core.sha(temp)!=manifest[name]:raise ValueError('Archive checksum mismatch: '+name)
    temp.replace(dest)
   if core.sha(dest)!=manifest[name]:raise ValueError('Conflicting recovered input: '+name)
 spec=json.loads((root/'run_spec.json').read_text())
 if spec['months'][0]!='2022-01-01' or spec['months'][-1]!='2026-05-01':raise ValueError('Not the completed full53 experiment')
 for role,name in [('model','expanded.ubj'),('encoder','encoder.joblib'),('module','post_rally_matched_test.py')]:
  if core.sha(root/'recovered/expanded'/name)!=core.HASHES[role]:raise ValueError('Wrong saved asset '+role)
 if core.sha(root/'recovered/production/march_golden.parquet')!=core.HASHES['march']:raise ValueError('Wrong March reference')
 if core.sha(root/'code/g1_saved_payoff_comparison_core.py')!=spec['core_sha256']:raise ValueError('Wrong scoring core')
 if core.sha(root/'code/g1_fannie_full53.py')!=spec['helper_sha256']:raise ValueError('Wrong archived helper')
 return spec

def members_cte():
 return f"""WITH target AS (
 SELECT pool_id,coupon pool_coupon,actual_coupon,issue_date FROM `{POOLS}` WHERE mapping_status='known_target'
 ), members AS (
 SELECT 'fannie' agency,CAST(a.loan_id AS STRING) loan_id,a.as_of_date feature_date,a.pool_id,
 p.pool_coupon,p.actual_coupon,p.issue_date,CAST(a.current_upb AS FLOAT64) current_upb
 FROM `{SERVICING}` a JOIN target p ON a.pool_id=p.pool_id
 WHERE a.as_of_date BETWEEN DATE '2026-06-01' AND DATE '2026-09-01'
 AND a.as_of_date=DATE_TRUNC(a.as_of_date,MONTH) AND a.orig_term=360
 AND a.current_upb>0 AND p.issue_date<=a.as_of_date)"""

def census_sql():
 return members_cte()+""" SELECT feature_date,pool_coupon,COUNT(*) member_rows,
 COUNT(DISTINCT loan_id) distinct_member_loans,SUM(current_upb) member_upb
 FROM members GROUP BY 1,2 ORDER BY 1,2"""

def freeze_sql(table):
 if not re.fullmatch(re.escape(PROJECT)+r'\.oasive_quant_sandbox\.g1_julsep_[a-z0-9_]+',table):raise ValueError('Writes restricted to unique sandbox g1_julsep_ tables')
 expressions=[]
 for c in core.BASE:
  e='f.'+c
  if c in ['occupancy','channel','property_type']:e=f"COALESCE(f.{c},'UNK')"
  if c=='num_units':e='COALESCE(f.num_units,1)'
  expressions.append(e+' AS '+c)
 return f"""CREATE TABLE `{table}` PARTITION BY feature_date CLUSTER BY pool_id,loan_id
 OPTIONS(expiration_timestamp=TIMESTAMP_ADD(CURRENT_TIMESTAMP(),INTERVAL 30 DAY),
 description='Saved October production versus saved expanded Fannie G1; June calibration sensitivity; Jul-Sep evaluation') AS
 """+members_cte()+f"""
 SELECT m.*,f.loan_id IS NOT NULL feature_record_present,CAST(f.current_upb AS FLOAT64) feature_upb,
 {', '.join(expressions)},
 f.occupancy IS NULL original_occupancy_missing,f.channel IS NULL original_channel_missing,
 f.property_type IS NULL original_property_type_missing,f.num_units IS NULL original_num_units_missing,
 l.loan_id IS NOT NULL label_record_present,l.is_curtailment IS NOT NULL label_rule_eligible,
 l.termination_component,
 IF(l.loan_id IS NULL,NULL,IF(l.termination_component IN('refinance','turnover','cashout','buyout'),1,0)) actual_payoff,
 IF(l.loan_id IS NULL,NULL,IF(l.termination_component IN('refinance','turnover','cashout','buyout','turnover_proxy','voluntary_unknown'),1,0)) broad_payoff
 FROM members m LEFT JOIN `{FEATURES}` f ON f.agency=m.agency AND f.loan_id=m.loan_id
 AND f.as_of_date=m.feature_date AND f.as_of_date BETWEEN DATE '2026-06-01' AND DATE '2026-09-01' AND f.orig_term=360
 LEFT JOIN `{LABELS}` l ON l.agency=m.agency AND l.loan_id=m.loan_id
 AND l.as_of_date=m.feature_date AND l.as_of_date BETWEEN DATE '2026-06-01' AND DATE '2026-09-01'"""

def month_sql(table,month):
 if month not in MONTHS:raise ValueError('Outside locked June-September calendar')
 where=f"feature_date=DATE '{month}' AND agency='fannie'"
 columns=', '.join(core.KEY+['pool_id','pool_coupon','current_upb']+core.BASE)
 return (f'SELECT * FROM `{table}` WHERE {where}',f"""SELECT agency,loan_id,feature_date,pool_id,pool_coupon,current_upb,
 (SELECT v.prob FROM UNNEST(predicted_is_payoff_probs) v WHERE SAFE_CAST(v.label AS INT64)=1) production_raw
 FROM ML.PREDICT(MODEL `{MODEL}`,(SELECT {columns} FROM `{table}` WHERE {where}))""")

def validate_inventory(c,a):
 c=old.normalized_dates(c);a=old.normalized_dates(a)
 expected={(mo,cp) for mo in MONTHS for cp in [6.,6.5]}
 for name,d in [('census',c),('panel',a)]:
  if d.duplicated(['feature_date','pool_coupon']).any() or set(zip(d.feature_date,d.pool_coupon))!=expected:raise ValueError(name+' must cover all 8 June-September coupon cells')
 if not c.member_rows.eq(c.distinct_member_loans).all() or not c.member_rows.gt(0).all():raise ValueError('Empty or duplicate servicing membership')
 z=c.merge(a,on=['feature_date','pool_coupon'],validate='one_to_one')
 if not z.member_rows.eq(z.panel_rows).all() or not z.panel_rows.eq(z.distinct_panel_loans).all():raise ValueError('Feature/label joins multiplied or lost loans')
 if not np.allclose(z.member_upb.to_numpy(float),z.panel_upb.to_numpy(float),atol=.1,rtol=1e-12):raise ValueError('Panel balances differ from census')
 flags=['missing_features','missing_labels','label_definition_gaps','balance_conflicts']
 if not z[flags].eq(0).all().all():raise ValueError('Resolve genuine feature/label/definition/balance errors; inspect panel_audit_raw.csv')
 return z

def assert_sources(client,pinned):
 for name,identity in pinned.items():
  if core.table_id(client.get_table(name))!=identity:raise ValueError('Source changed during freeze preparation: '+name)

def freeze(client,root,prep):
 from google.cloud import bigquery
 root=Path(root);panel=prep['panel'];record=root/'frozen_panel_identity.json'
 if record.exists():
  identity=json.loads(record.read_text())
  if core.table_id(client.get_table(panel))!=identity:raise ValueError('Frozen panel changed or expired; recover saved monthly files or use a new run')
  return identity
 assert_sources(client,prep['source_identities'])
 query=freeze_sql(panel);marker=root/'freeze_job.json';digest=hashlib.sha256(query.encode()).hexdigest()
 if marker.exists():
  saved=json.loads(marker.read_text())
  if saved['sql_sha256']!=digest:raise ValueError('Freeze SQL changed')
  job=client.get_job(saved['job_id'],location='us-central1')
 else:
  job=client.query(query,job_config=bigquery.QueryJobConfig(maximum_bytes_billed=QUERY_CAP),location='us-central1')
  wjson(marker,dict(job_id=job.job_id,sql_sha256=digest,panel=panel))
 print('Freeze job:',job.job_id,flush=True);job.result();assert_sources(client,prep['source_identities'])
 identity=core.table_id(client.get_table(panel));wjson(record,identity);return identity

def assert_run(client,root,spec):
 root=Path(root)
 if core.sha(__file__)!=spec['helper_sha256'] or core.sha(core.__file__)!=spec['core_sha256']:raise ValueError('Scoring code changed')
 if core.sha(old.__file__)!=spec['archived_helper_sha256']:raise ValueError('Archived validator changed')
 if core.table_id(client.get_table(spec['panel']))!=spec['panel_identity']:raise ValueError('Frozen panel changed or expired')
 if core.model_id(client.get_model(MODEL))!=spec['production_identity']:raise ValueError('October production artifact changed')
 for role,path in spec['asset_paths'].items():
  if core.sha(path)!=core.HASHES[role]:raise ValueError('Saved asset changed: '+role)

def score_month(client,root,spec,month,inventory,model,encoder,module):
 root=Path(root);assert_run(client,root,spec)
 folder=root/'months'/month;folder.mkdir(parents=True,exist_ok=True);done=folder/'scoring_complete.json'
 if done.exists():
  d=json.loads(done.read_text())
  if d['spec_sha256']!=core.sha(root/'run_spec.json'):raise ValueError('Changed run specification')
  for name,digest in d['files'].items():
   if core.sha(folder/name)!=digest:raise ValueError('Changed saved prediction file')
  print('Reusing verified saved month:',month);return pd.read_parquet(folder/'paired_predictions_before.parquet')
 input_sql,predict_sql=month_sql(spec['panel'],month)
 f=old.validate_month(core.cached_query(client,'input_'+month,input_sql,root),month,inventory)
 pred=core.cached_query(client,'predict_'+month,predict_sql,root);raw=pred.pop('production_raw').to_numpy(float)
 if not np.isfinite(raw).all() or ((raw<0)|(raw>1)).any():raise ValueError('Invalid production probabilities')
 pred['production_before']=.12*raw/(1-raw+.12*raw)
 paired=core.join_predictions(f,pred,core.score_expanded(model,encoder,module,f))
 paired.to_parquet(folder/'paired_predictions_before.parquet',index=False)
 cats=module.cats(f);rows=[]
 for i,col in enumerate(core.CAT):
  unknown=~cats[col].isin(encoder.categories_[i]);rows.append(dict(month=month,feature=col,unknown_rows=int(unknown.sum()),unknown_upb_share=float(f.loc[unknown,'current_upb'].sum()/f.current_upb.sum())))
 pd.DataFrame(rows).to_csv(folder/'category_coverage.csv',index=False)
 wjson(done,dict(spec_sha256=core.sha(root/'run_spec.json'),files={p.name:core.sha(p) for p in folder.iterdir() if p.is_file() and p.name!='scoring_complete.json'}))
 assert_run(client,root,spec);return paired

def odds(p,factor):
 p=np.clip(np.asarray(p,dtype=float),1e-12,1-1e-12)
 if not np.isfinite(factor) or factor<=0:raise ValueError('Invalid odds factor')
 return factor*p/(1-p+factor*p)

def june_calibration(june,module):
 june=core.normalize(june)
 if not june.feature_date.eq(CAL_MONTH).all():raise ValueError('Calibration must use June only; July-September outcomes forbidden')
 y=june.actual_payoff.to_numpy(float);w=june.current_upb.to_numpy(float)
 if not set(np.unique(y))=={0,1} or not np.isfinite(w).all() or (w<=0).any():raise ValueError('Invalid calibration outcomes or weights')
 factors={};rows=[]
 for arm in ['production','expanded']:
  p=june[arm+'_before'].to_numpy(float)
  factor=module.solve_factor(p,y,w);factors[arm]=factor
  rows.append(dict(month=CAL_MONTH,agency='fannie',arm=arm,scope='all selected G1 coupons combined',loans=len(june),payoff_events=int(y.sum()),upb=float(w.sum()),actual_pct=100*float(np.average(y,weights=w)),before_pct=100*float(np.average(p,weights=w)),after_pct=100*float(np.average(odds(p,factor),weights=w)),factor_on_before_probability=factor,equivalent_factor_on_raw_probability=.12*factor))
 return factors,pd.DataFrame(rows)

def report(f,month,stage):
 if month not in EVAL_MONTHS:raise ValueError('June must not enter evaluation results')
 out=core.group_report(f,month);extra=[]
 # Whole-book small balances, plus balance by payment status: not restricted to young/high-incentive loans.
 for coupon in [6.,6.5]:
  cp=f.pool_coupon.eq(coupon);size=pd.to_numeric(f.orig_upb);dlq=pd.to_numeric(f.dlq_days)
  groups=[('small_all_ages',str(coupon)+'_lt150k',cp&size.lt(150000))]
  for label,mask in [('lt150k',size.lt(150000)),('150_300k',size.ge(150000)&size.lt(300000)),('300_500k',size.ge(300000)&size.lt(500000)),('500kplus',size.ge(500000))]:
   for status,s in [('current',dlq.eq(0)),('delinquent',dlq.gt(0)),('unknown',dlq.isna()|dlq.lt(0))]:groups.append(('status_balance',f'{coupon}_{status}_{label}',cp&mask&s))
  for key,name,mask in groups:
   if mask.any():extra.append(dict(month=month,grouping=key,group=name,**core.metric(f.loc[mask])))
 out=pd.concat([out,pd.DataFrame(extra)],ignore_index=True)
 out['stage']=stage;out['agency']='fannie';out['role']='evaluation'
 return out

def report_stages(f,month,factors):
 if month not in EVAL_MONTHS:raise ValueError('Evaluation is July-September only')
 frames=[]
 for stage in STAGES:
  x=f.copy()
  if stage!='before':
   for arm in ['production','expanded']:
    factor=factors[arm] if stage=='june_separate' else factors['production']
    x[arm+'_before']=odds(f[arm+'_before'],factor)
  frames.append(report(x,month,stage))
 return pd.concat(frames,ignore_index=True)

def summarize(results,maturity,june_review=None):
 if not set(results.month).issubset(EVAL_MONTHS):raise ValueError('Evaluation summary includes non-evaluation month')
 if maturity.duplicated('month').any() or set(maturity.month)!=set(EVAL_MONTHS):raise ValueError('Require one maturity review row per evaluation month')
 if not maturity.status.isin(['unconfirmed','provisional','confirmed']).all():raise ValueError('Invalid maturity status')
 if maturity.loc[maturity.status.eq('confirmed'),'evidence'].fillna('').str.strip().eq('').any():raise ValueError('Confirmed outcome status requires recorded source-completeness evidence')
 review=june_review or {'status':'unconfirmed','evidence':''}
 if review['status'] not in ['unconfirmed','provisional','confirmed']:raise ValueError('Invalid June maturity status')
 if review['status']=='confirmed' and not review.get('evidence','').strip():raise ValueError('Confirmed June outcomes require evidence')
 results=results.merge(maturity.rename(columns={'status':'outcome_status','evidence':'outcome_evidence'}),on='month',validate='many_to_one')
 results['calibration_outcome_status']=np.where(results.stage.eq('before'),'not_applicable',review['status'])
 rows=[]
 for name,months in [('jul_aug',EVAL_MONTHS[:2]),('jul_sep',EVAL_MONTHS)]:
  for key,g in results.loc[results.month.isin(months)].groupby(['stage','grouping','group'],observed=True):
   w=g.upb.to_numpy(float);avg=lambda c:float(np.average(g[c],weights=w))
   d=dict(period=name,stage=key[0],grouping=key[1],group=key[2],months=g.month.nunique(),expected_months=len(months),loan_months=int(g.loans.sum()),upb_month_exposure_bn=float(w.sum()/1e9),all_outcomes_confirmed=bool(g.outcome_status.eq('confirmed').all()))
   d['calibration_outcomes_confirmed']=key[0]=='before' or review['status']=='confirmed'
   d['ready_for_final_interpretation']=d['all_outcomes_confirmed'] and d['calibration_outcomes_confirmed']
   for arm in ['production','expanded']:
    d[arm+'_weighted_abs_bias_pp']=float(np.average(abs(g[arm+'_bias_pp']),weights=w));d[arm+'_weighted_signed_bias_pp']=avg(arm+'_bias_pp')
   d['weighted_abs_bias_gain_pp']=d['production_weighted_abs_bias_pp']-d['expanded_weighted_abs_bias_pp']
   for loss in ['brier','logloss']:d['weighted_'+loss+'_gain']=avg(loss+'_gain')
   rows.append(d)
 return results,pd.DataFrame(rows)


Writing /content/g1_julsep_saved_models_20261101.py


## Step 4 — Load exact scoring code and replay March
March replay checks the saved expanded model, encoder and sparse matrix. It does not select the later-month population. Production is scored in BigQuery later.


In [7]:
import numpy as np,pandas as pd,joblib,xgboost as xgb

def load_exact(name,path):
    spec=importlib.util.spec_from_file_location(name,str(path));mod=importlib.util.module_from_spec(spec)
    sys.modules[name]=mod;spec.loader.exec_module(mod);return mod
core=load_exact('g1_saved_payoff_comparison_core',INPUTS/'code/g1_saved_payoff_comparison_core.py')
old=load_exact('g1_fannie_full53',INPUTS/'code/g1_fannie_full53.py')
follow=load_exact('g1_julsep_saved_models_20261101',Path('/content/g1_julsep_saved_models_20261101.py'))
assert core.sha(core.__file__)==OLD_SPEC['core_sha256']
assert core.sha(old.__file__)==OLD_SPEC['helper_sha256']
assert 'FROM raw_members WHERE has_prior' not in old.members_cte()
assert 'OR label_rule_eligible IS NOT TRUE OR actual_payoff IS NULL' not in old.panel_audit_sql('p.d.t')
EP=INPUTS/'recovered/expanded';PP=INPUTS/'recovered/production'
ASSET_PATHS={'model':str(EP/'expanded.ubj'),'encoder':str(EP/'encoder.joblib'),
             'module':str(EP/'post_rally_matched_test.py'),'march':str(PP/'march_golden.parquet')}
for role,path in ASSET_PATHS.items():assert core.sha(path)==core.HASHES[role],'Wrong saved asset: '+role
m=load_exact('julsep_exact_saved_matrix',ASSET_PATHS['module'])
assert m.BASE==core.BASE and m.CAT==core.CAT and m.Q==.12
EXP_SPEC=json.loads((EP/'experiment_spec.json').read_text())
assert EXP_SPEC['features']==core.BASE and EXP_SPEC['config']['seed']==20261021
assert EXP_SPEC['config']['new_timing_features'] is False and EXP_SPEC['config']['loss_weights']=='one'
ENCODER=joblib.load(EP/'encoder.joblib')
EXPANDED=xgb.Booster(model_file=str(EP/'expanded.ubj'));EXPANDED.set_param({'nthread':2,'device':'cpu'})
golden=core.normalize(pd.read_parquet(PP/'march_golden.parquet'))
assert len(golden)==273915
replay=core.score_expanded(EXPANDED,ENCODER,m,golden)
assert np.allclose(replay,golden.expanded_before,atol=1e-7,rtol=0),'March replay mismatch: stop'
core.write_json(ROOT/'saved_model_replay.json',dict(loans=len(golden),max_error=float(np.max(abs(replay-golden.expanded_before)))))
CODE=ROOT/'code';CODE.mkdir(exist_ok=True)
shutil.copy2(follow.__file__,CODE/Path(follow.__file__).name)
del golden,replay;gc.collect()
print('Exact saved expanded model verified. No training performed.')


Exact saved expanded model verified. No training performed.


## Step 5 — Authenticate and pin source/model identities
Use the existing Google account. No bucket. Source snapshots and corrected pool metadata must match the old experiment. Any servicing refresh is recorded; its newly pinned identity must stay unchanged during preparation. The NEW specification correctly records no prior-month eligibility requirement; original ZIPs are unchanged.


In [8]:
from google.colab import auth
from google.cloud import bigquery
auth.authenticate_user()
CLIENT=bigquery.Client(project=follow.PROJECT,location='us-central1')
PROD_ID=core.model_id(CLIENT.get_model(follow.MODEL))
assert PROD_ID==OLD_SPEC['production_identity'],'Wrong or changed October production artifact'
prep_path=ROOT/'preparation_spec.json'
if prep_path.exists():
    PREP=json.loads(prep_path.read_text())
    assert PREP['helper_sha256']==core.sha(follow.__file__) and PREP['core_sha256']==core.sha(core.__file__)
    assert PREP['production_identity']==PROD_ID and PREP['months']==follow.MONTHS
else:
    PINNED={s:core.table_id(CLIENT.get_table(s)) for s in follow.SOURCES}
    for name in follow.SOURCES:assert CLIENT.get_table(name).location.lower()=='us-central1'
    for name in [follow.FEATURES,follow.LABELS,follow.POOLS]:
        assert PINNED[name]==OLD_SPEC['source_identities'][name],'Original snapshot or roster changed: '+name
    PREP=dict(version='g1_saved_julsep_v1',months=follow.MONTHS,evaluation_months=follow.EVAL_MONTHS,
              calibration_month=follow.CAL_MONTH,panel=follow.PROJECT+'.oasive_quant_sandbox.g1_julsep_fannie_'+uuid.uuid4().hex[:12],
              source_identities=PINNED,production_identity=PROD_ID,helper_sha256=core.sha(follow.__file__),
              core_sha256=core.sha(core.__file__),prior_month_required=False,
              payoff_evaluation_rule='Present binary payoff outcome; curtailment training eligibility diagnostic only',
              june_sensitivity='One UPB-weighted intercept per arm over combined selected Fannie G1; June only; fixed thereafter',
              primary_stage='before',independent_blind_test=False)
    core.write_json(prep_path,PREP)
if not (ROOT/'frozen_panel_identity.json').exists():follow.assert_sources(CLIENT,PREP['source_identities'])
metadata=[]
for name in follow.SOURCES:
    t=CLIENT.get_table(name)
    metadata.append(dict(table=name,table_type=t.table_type,rows=t.num_rows,modified=str(t.modified),created=str(t.created),
                         matches_old_source=PREP['source_identities'][name]==OLD_SPEC['source_identities'][name]))
pd.DataFrame(metadata).to_csv(ROOT/'source_metadata.csv',index=False)
print(pd.DataFrame(metadata).to_string(index=False))
print('Only persistent BigQuery output:',PREP['panel'])


                                                                                           table table_type       rows                         modified                          created  matches_old_source
            gen-lang-client-0343560978.oasive_quant_bq.feature_loan_month_mat_bak20261005_171029   SNAPSHOT 1461751273 2026-10-05 17:10:38.346000+00:00 2026-10-05 17:10:38.346000+00:00                True
             gen-lang-client-0343560978.oasive_quant_bq.label_loan_month_fast_bak20261010_193027   SNAPSHOT 2004931416 2026-10-10 19:30:28.047000+00:00 2026-10-10 19:30:28.047000+00:00                True
                        gen-lang-client-0343560978.oasive_quant_bq.fact_loan_monthly_fannie_sftp      TABLE  852158282 2026-10-10 12:46:46.818000+00:00 2026-09-28 09:16:40.876000+00:00                True
gen-lang-client-0343560978.oasive_quant_sandbox.g1_original_components_20261016_v2_pool_metadata      TABLE      31545 2026-10-17 00:21:35.329000+00:00 2026-10-17 00:21:35.329000+0

## Step 6 — Verify roster and dry-run queries
Expected known-target pools: 11,599 for this package. Reconstruct each month's membership, not March survivors. Review byte estimates before Step 7. Dry runs do not execute the census or freeze. Per preparation query cap: 500 GiB; total planned cap: 750 GiB including later reads. Estimates are not quoted bills; storage is additional. Do not change population/features after outcomes to obtain a preferred result.


In [9]:
roster=old.cached_query(CLIENT,'pool_roster',old.roster_sql(),ROOT)
target=old.validate_roster(roster)
archived=pd.read_csv(INPUTS/'pool_roster_audit.csv',dtype={'pool_id':str})
keys=['pool_id','mapping_status','pool_coupon']
a=roster[keys].copy();b=archived[keys].copy()
for frame in [a,b]:
    frame['pool_id']=frame.pool_id.astype(str)
    frame['mapping_status']=frame.mapping_status.fillna('<missing>').astype(str)
    frame['pool_coupon']=pd.to_numeric(frame.pool_coupon,errors='raise').astype(float)
assert a.sort_values('pool_id').reset_index(drop=True).equals(b.sort_values('pool_id').reset_index(drop=True)),'Changed corrected roster'
assert len(target)==11599,'Unexpected target-pool scope'
roster.to_csv(ROOT/'pool_roster_audit.csv',index=False)
SQL=ROOT/'sql';SQL.mkdir(exist_ok=True)
QUERIES={'membership_census':follow.census_sql(),'freeze_panel':follow.freeze_sql(PREP['panel'])}
rows=[]
for name,q in QUERIES.items():
    (SQL/(name+'.sql')).write_text(q)
    n=old.dry(CLIENT,q,follow.QUERY_CAP)
    rows.append(dict(query=name,estimated_bytes=n,estimated_GiB=n/1024**3))
ESTIMATES=pd.DataFrame(rows)
assert ESTIMATES.estimated_bytes.sum()<=follow.PLAN_CAP,'Review excessive estimate'
ESTIMATES.to_csv(ROOT/'preparation_estimates.csv',index=False)
print(ESTIMATES.to_string(index=False))
print('Preparation estimate GiB:',ESTIMATES.estimated_bytes.sum()/1024**3)


Query: pool_roster job: 7e43e1f5-2121-44c1-8383-f2d863b8fb9e
            query  estimated_bytes  estimated_GiB
membership_census      46870076459      43.651160
     freeze_panel     137863028809     128.394951
Preparation estimate GiB: 172.04611121490598


## Step 7 — Run the independent membership census
Actual BigQuery scan. Require four months × two coupons = eight positive cells with unique IDs. June is calibration-only. Counts need not equal March. Do not require future survival.


In [10]:
if not (ROOT/'frozen_panel_identity.json').exists():follow.assert_sources(CLIENT,PREP['source_identities'])
CENSUS=old.normalized_dates(old.cached_query(CLIENT,'membership_census',QUERIES['membership_census'],ROOT,limit=follow.QUERY_CAP))
assert not CENSUS.duplicated(['feature_date','pool_coupon']).any()
assert set(zip(CENSUS.feature_date,CENSUS.pool_coupon))=={(mo,cp) for mo in follow.MONTHS for cp in [6.,6.5]}
assert CENSUS.member_rows.gt(0).all() and CENSUS.member_rows.eq(CENSUS.distinct_member_loans).all()
CENSUS.to_csv(ROOT/'membership_census.csv',index=False)
print(CENSUS.to_string(index=False))


Source audit query: membership_census job: c3faaca4-b4f4-42d0-ac61-f07a2416c0d9
feature_date  pool_coupon  member_rows  distinct_member_loans   member_upb
  2026-06-01          6.0       177817                 177817 5.301220e+10
  2026-06-01          6.5        77203                  77203 2.143465e+10
  2026-07-01          6.0       175387                 175387 5.218070e+10
  2026-07-01          6.5        75709                  75709 2.095224e+10
  2026-08-01          6.0       173145                 173145 5.140184e+10
  2026-08-01          6.5        74307                  74307 2.049127e+10
  2026-09-01          6.0       171257                 171257 5.076149e+10
  2026-09-01          6.5        73121                  73121 2.010918e+10


## Step 8 — Freeze the matched sandbox panel
Large actual query. All 25 features, same payoff rule and current-UPB weights. Never overwrites production or the old panel. New panel expires after 30 days. Closing Colab does not delete it sooner.


In [11]:
PANEL_ID=follow.freeze(CLIENT,ROOT,PREP)
print('Frozen panel:',PREP['panel'])
print('Panel identity saved; rows:',PANEL_ID.get('numRows'))


Freeze job: 30d856ec-218a-4e2a-9c85-fec9d4ff525a
Frozen panel: gen-lang-client-0343560978.oasive_quant_sandbox.g1_julsep_fannie_3c941ffb1f82
Panel identity saved; rows: 997946


## Step 9 — Validate features, outcomes, balances and coverage
All eight cells must reconcile with the census. Missing labels/features, duplicate joins, broad/narrow payoff differences or balance disagreement stop. Null curtailment training eligibility is diagnostic, not exclusion.

Label presence does not prove all source releases/corrections arrived. September is especially important: the inherited label snapshot was taken in October. Review counts and rates; never fill missing labels with zeros or silently switch to live/revised labels.


In [12]:
audit_sql=old.panel_audit_sql(PREP['panel'])
old.dry(CLIENT,audit_sql,follow.REPORT_CAP)
PANEL_AUDIT=core.cached_query(CLIENT,'panel_audit',audit_sql,ROOT)
PANEL_AUDIT.to_csv(ROOT/'panel_audit_raw.csv',index=False)
print(PANEL_AUDIT.to_string(index=False))
INVENTORY=follow.validate_inventory(CENSUS,PANEL_AUDIT)
INVENTORY.to_csv(ROOT/'verified_inventory_8_cells.csv',index=False)
coverage=INVENTORY[['feature_date','pool_coupon','member_rows','member_upb','payoff_events','missing_labels']].copy()
coverage['payoff_count_rate_pct']=100*coverage.payoff_events/coverage.member_rows
coverage.to_csv(ROOT/'monthly_outcome_coverage.csv',index=False)
print('Genuine source checks passed. Outcome completeness is reviewed separately.')


Query: panel_audit job: 1ad8b911-065f-4b7a-8e12-fd8c3d3a1f8f
feature_date  pool_coupon  panel_rows  distinct_panel_loans    panel_upb  payoff_events  missing_features  missing_labels  training_rule_ineligible_rows  training_rule_ineligible_payoffs  training_rule_ineligible_upb  label_definition_gaps  balance_conflicts
  2026-06-01          6.0      177817                177817 5.301220e+10           2430                 0               0                              0                                 0                           0.0                      0                  0
  2026-06-01          6.5       77203                 77203 2.143465e+10           1494                 0               0                              0                                 0                           0.0                      0                  0
  2026-07-01          6.0      175387                175387 5.218070e+10           2242                 0               0                              0          

## Step 10 — Record outcome maturity and lock scoring
Default July/August status is `unconfirmed`; September `provisional`. Scoring is allowed, but outputs preserve these labels. Label presence alone cannot make a result final.

Mark a month `confirmed` only with evidence of complete relevant servicing releases and payoff terminations/corrections. Record a batch ID, reconciliation file or documented completeness check in `evidence`. Last-modified dates alone are insufficient. Leave defaults if evidence is unavailable. June calibration has its own completeness review; an unconfirmed June source keeps calibrated conclusions provisional even if later months are confirmed. Do not change labels to favor a model.

The scoring specification locks artifacts, source panel, features and corrections. Changes need a NEW run directory. No tree is refitted.


In [13]:
maturity_path=ROOT/'outcome_maturity_review.csv'
if not maturity_path.exists():
    pd.DataFrame([dict(month=mo,status='provisional' if mo=='2026-09-01' else 'unconfirmed',evidence='')
                  for mo in follow.EVAL_MONTHS]).to_csv(maturity_path,index=False)
# Optional: edit this CSV only with documented completeness evidence.
MATURITY=pd.read_csv(maturity_path,keep_default_na=False)
# Fill only after verifying source completeness. Leave {} if evidence is unavailable.
MATURITY_UPDATES={}
# Example structure (replace with actual evidence before using):
# MATURITY_UPDATES={'2026-07-01':{'status':'confirmed','evidence':'Actual source batch/reconciliation reference'}}
for month,review in MATURITY_UPDATES.items():
    assert month in follow.EVAL_MONTHS and review['status'] in ['unconfirmed','provisional','confirmed']
    assert review['status']!='confirmed' or review.get('evidence','').strip(),'Evidence required'
    MATURITY.loc[MATURITY.month.eq(month),['status','evidence']]=[review['status'],review.get('evidence','')]
MATURITY.to_csv(maturity_path,index=False)
june_review_path=ROOT/'june_outcome_review.json'
if not june_review_path.exists():core.write_json(june_review_path,dict(status='unconfirmed',evidence=''))
JUNE_OUTCOME_UPDATE=None  # Optional dict(status='confirmed', evidence='<actual completeness evidence>')
if JUNE_OUTCOME_UPDATE is not None:core.write_json(june_review_path,JUNE_OUTCOME_UPDATE)
JUNE_REVIEW=json.loads(june_review_path.read_text())
display(MATURITY)
print('June outcome review:',JUNE_REVIEW)
SCORING=[]
for month in follow.MONTHS:
    for role,q in zip(['input','predict'],follow.month_sql(PREP['panel'],month)):
        (SQL/(role+'_'+month+'.sql')).write_text(q)
        n=old.dry(CLIENT,q,follow.REPORT_CAP)
        SCORING.append(dict(month=month,role=role,estimated_bytes=n,estimated_GiB=n/1024**3))
SCORING=pd.DataFrame(SCORING);SCORING.to_csv(ROOT/'scoring_estimates.csv',index=False)
assert ESTIMATES.estimated_bytes.sum()+SCORING.estimated_bytes.sum()<=follow.PLAN_CAP
SPEC=dict(version='g1_saved_julsep_v1',panel=PREP['panel'],panel_identity=PANEL_ID,production_identity=PROD_ID,
          months=follow.MONTHS,evaluation_months=follow.EVAL_MONTHS,calibration_month=follow.CAL_MONTH,
          asset_paths=ASSET_PATHS,helper_sha256=core.sha(follow.__file__),core_sha256=core.sha(core.__file__),
          archived_helper_sha256=core.sha(old.__file__),source_identities=PREP['source_identities'],
          feature_count=25,sampling_odds_multiplier=.12,prior_month_required=False,
          calibration_design=PREP['june_sensitivity'],primary_stage='before',
          model_training_cutoff='2026-05-31',independent_blind_test=False,
          original_run_spec_sha256=core.sha(INPUTS/'run_spec.json'),saved_model_replay_sha256=core.sha(ROOT/'saved_model_replay.json'))
lock=ROOT/'run_spec.json'
if lock.exists():assert json.loads(lock.read_text())==SPEC,'Changed experiment: use a NEW run directory'
else:core.write_json(lock,SPEC)
print(SCORING.to_string(index=False))


,month,status,evidence
0,2026-07-01,unconfirmed,
1,2026-08-01,unconfirmed,
2,2026-09-01,provisional,


June outcome review: {'status': 'unconfirmed', 'evidence': ''}
     month    role  estimated_bytes  estimated_GiB
2026-06-01   input         95652582       0.089083
2026-06-01 predict         85180546       0.079331
2026-07-01   input         94178392       0.087710
2026-07-01 predict         83893594       0.078132
2026-08-01   input         92806352       0.086433
2026-08-01 predict         82698554       0.077019
2026-09-01   input         91623454       0.085331
2026-09-01 predict         81690386       0.076080


## Step 11 — Score June and fit its separate calibration sensitivity
One factor per model over **all selected June Fannie G1 loans, both coupons combined**. No fitting by size, age, coupon or delinquency, and no July–September outcomes.

Before probabilities already include the 12% negative-sampling correction. The June factor is an **additional multiplier on corrected odds**, not a second application of 0.12. `equivalent_factor_on_raw_probability` = 0.12 × additional factor.

Use the archived module's `solve_factor` method, not old small-sample factors. This research sensitivity does not claim to reproduce deployed production calibration. If June outcome completeness is unestablished, treat the sensitivity as provisional too.


In [14]:
JUNE=follow.score_month(CLIENT,ROOT,SPEC,'2026-06-01',INVENTORY,EXPANDED,ENCODER,m)
assert (JUNE.actual_payoff.value_counts().reindex([0,1],fill_value=0)>=30).all(),'Insufficient June events'
FACTORS,CALIBRATION=follow.june_calibration(JUNE,m)
CALIBRATION.to_csv(ROOT/'june_calibration_sensitivity.csv',index=False)
FACTOR_LOCK=dict(run_spec_sha256=core.sha(ROOT/'run_spec.json'),june_predictions_sha256=core.sha(ROOT/'months/2026-06-01/paired_predictions_before.parquet'),
                 factors=FACTORS,fit_month='2026-06-01',scope='overall matched Fannie G1; combined coupons',applied_to='July-September only',
                 factor_scale='additional odds multiplier on already .12-corrected probability',production_calibration_replica=False)
factor_path=ROOT/'june_factors.json'
if factor_path.exists():assert json.loads(factor_path.read_text())==FACTOR_LOCK,'June factor changed: stop'
else:core.write_json(factor_path,FACTOR_LOCK)
print(CALIBRATION.to_string(index=False))
del JUNE;gc.collect()


Query: input_2026-06-01 job: 96c9b748-6937-4b65-b364-b12e554014a1
Query: predict_2026-06-01 job: 08256fbb-d6a8-4553-a427-f0b629dd8c1e
     month agency        arm                            scope  loans  payoff_events          upb  actual_pct  before_pct  after_pct  factor_on_before_probability  equivalent_factor_on_raw_probability
2026-06-01 fannie production all selected G1 coupons combined 255020           3924 7.444685e+10     1.57159    1.807301    1.57159                      0.866866                              0.104024
2026-06-01 fannie   expanded all selected G1 coupons combined 255020           3924 7.444685e+10     1.57159    1.838158    1.57159                      0.850551                              0.102066


36226

## Step 12 — Score July, August, September and report fixed stages
**before:** main result with the same treatment as full53. **june_separate:** each model's own June-only factor. **june_common_production:** production's June factor applied to both, a diagnostic.

Same loans and weights in each arm/stage per month; same 25-feature inputs. Category coverage records unseen encoder categories; never refit the encoder. Cache predictions with checksums. Do not alter features, weights, factors or population based on later results.


In [15]:
assert json.loads((ROOT/'june_factors.json').read_text())==FACTOR_LOCK
frames=[]
for month in follow.EVAL_MONTHS:
    paired=follow.score_month(CLIENT,ROOT,SPEC,month,INVENTORY,EXPANDED,ENCODER,m)
    frame=follow.report_stages(paired,month,FACTORS)
    frame.to_csv(ROOT/'months'/month/'accuracy_all_stages.csv',index=False)
    frames.append(frame)
    print(month,'loans:',len(paired),'events:',int(paired.actual_payoff.sum()))
    del paired;gc.collect()
RESULTS=pd.concat(frames,ignore_index=True)
MATURITY=pd.read_csv(maturity_path,keep_default_na=False)
RESULTS,PERIODS=follow.summarize(RESULTS,MATURITY,JUNE_REVIEW)
RESULTS.to_csv(ROOT/'all_evaluation_groups.csv',index=False)
PERIODS.to_csv(ROOT/'period_summary.csv',index=False)
core.write_json(ROOT/'completion.json',dict(evaluation_months=follow.EVAL_MONTHS,calibration_month='2026-06-01',
                                         models_retrained=False,source_panel=PREP['panel'],primary_stage='before',
                                         independent_blind_test=False,all_evaluation_outcomes_confirmed=bool(MATURITY.status.eq('confirmed').all())))


Query: input_2026-07-01 job: 40d6c6db-79ba-4f0c-9c4e-4bc497412c23
Query: predict_2026-07-01 job: 63b057b1-6eb0-4a0a-9b86-e25634a81045
2026-07-01 loans: 251096 events: 3644
Query: input_2026-08-01 job: 354837f4-bcd5-4659-aafb-c8a2de531579
Query: predict_2026-08-01 job: 69e28432-1ad8-42e5-8a70-33e64923a92c
2026-08-01 loans: 247452 events: 3074
Query: input_2026-09-01 job: 4a8371b8-42ca-43b7-b2ca-d802de2bf32e
Query: predict_2026-09-01 job: b88654c4-98d5-4da2-a155-b138210677f8
2026-09-01 loans: 244378 events: 0


## Step 13 — Review monthly/period comparisons
Positive gains favor expanded. Review aggregate absolute error, Brier and log loss together. Use July/August separately from the provisional September-inclusive summary.

Period errors are monthly absolute errors weighted by monthly UPB, not errors after months cancel. UPB-month exposure and loan-month counts repeat loans across time; they are not distinct loan balances or independent borrowers.


In [16]:
cols=['month','stage','group','loans','payoff_events','actual_payoff_pct','production_predicted_pct','expanded_predicted_pct',
      'production_bias_pp','expanded_bias_pp','abs_bias_gain_pp','brier_gain','logloss_gain','outcome_status','calibration_outcome_status']
OVERALL=RESULTS.loc[RESULTS.grouping.eq('overall')].sort_values(['stage','month'])
COUPON=RESULTS.loc[RESULTS.grouping.eq('coupon')].sort_values(['stage','month','group'])
OVERALL.to_csv(ROOT/'monthly_overall.csv',index=False);COUPON.to_csv(ROOT/'monthly_coupon.csv',index=False)
print('MAIN RESULT: before additional calibration')
display(OVERALL.loc[OVERALL.stage.eq('before'),cols])
print('JUNE CALIBRATION SENSITIVITIES: separate results')
display(OVERALL.loc[~OVERALL.stage.eq('before'),cols])
display(PERIODS.loc[PERIODS.grouping.isin(['overall','coupon'])])


MAIN RESULT: before additional calibration


,month,stage,group,loans,payoff_events,actual_payoff_pct,production_predicted_pct,expanded_predicted_pct,production_bias_pp,expanded_bias_pp,abs_bias_gain_pp,brier_gain,logloss_gain,outcome_status,calibration_outcome_status
0,2026-07-01,before,all,251096,3644,1.498449,1.641454,1.704266,0.143004,0.205817,-0.062812,-0.000006,-0.000169,unconfirmed,not_applicable
537,2026-08-01,before,all,247452,3074,1.222574,1.608980,1.611009,0.386406,0.388436,-0.002030,0.000034,0.000280,unconfirmed,not_applicable
1050,2026-09-01,before,all,244378,0,0.000000,1.273057,1.296543,1.273057,1.296543,-0.023486,-0.000124,-0.000313,provisional,not_applicable


JUNE CALIBRATION SENSITIVITIES: separate results


,month,stage,group,loans,payoff_events,actual_payoff_pct,production_predicted_pct,expanded_predicted_pct,production_bias_pp,expanded_bias_pp,abs_bias_gain_pp,brier_gain,logloss_gain,outcome_status,calibration_outcome_status
358,2026-07-01,june_common_production,all,251096,3644,1.498449,1.427018,1.484083,-0.071431,-0.014367,0.057064,0.000015,-0.000083,unconfirmed,unconfirmed
879,2026-08-01,june_common_production,all,247452,3074,1.222574,1.398706,1.402358,0.176132,0.179784,-0.003652,0.000045,0.000285,unconfirmed,unconfirmed
1392,2026-09-01,june_common_production,all,244378,0,0.000000,1.106196,1.128043,1.106196,1.128043,-0.021847,-0.000101,-0.000280,provisional,unconfirmed
179,2026-07-01,june_separate,all,251096,3644,1.498449,1.427018,1.456976,-0.071431,-0.041473,0.029958,0.000016,-0.000089,unconfirmed,unconfirmed
708,2026-08-01,june_separate,all,247452,3074,1.222574,1.398706,1.376680,0.176132,0.154106,0.022026,0.000046,0.000316,unconfirmed,unconfirmed
1221,2026-09-01,june_separate,all,244378,0,0.000000,1.106196,1.107317,1.106196,1.107317,-0.001121,-0.000092,-0.000068,provisional,unconfirmed


,period,stage,grouping,group,months,expected_months,loan_months,upb_month_exposure_bn,all_outcomes_confirmed,calibration_outcomes_confirmed,ready_for_final_interpretation,production_weighted_abs_bias_pp,production_weighted_signed_bias_pp,expanded_weighted_abs_bias_pp,expanded_weighted_signed_bias_pp,weighted_abs_bias_gain_pp,weighted_brier_gain,weighted_logloss_gain
22,jul_aug,before,coupon,6.0,2,2,348532,103.582539,False,True,False,0.277366,0.277366,0.354749,0.354749,-0.077384,0.000004,-0.000047
23,jul_aug,before,coupon,6.5,2,2,150016,41.443507,False,True,False,0.229421,0.229421,0.150373,0.150373,0.079048,0.000037,0.000305
38,jul_aug,before,overall,all,2,2,498548,145.026046,False,True,False,0.263665,0.263665,0.296346,0.296346,-0.032681,0.000014,0.000054
201,jul_aug,june_common_production,coupon,6.0,2,2,348532,103.582539,False,False,False,0.120997,0.088078,0.157294,0.157294,-0.036297,0.000021,0.000058
202,jul_aug,june_common_production,coupon,6.5,2,2,150016,41.443507,False,False,False,0.129176,-0.040649,0.122023,-0.106610,0.007153,0.000053,0.000202
217,jul_aug,june_common_production,overall,all,2,2,498548,145.026046,False,False,False,0.123334,0.051292,0.096369,0.081879,0.026965,0.000030,0.000099
380,jul_aug,june_separate,coupon,6.0,2,2,348532,103.582539,False,False,False,0.120997,0.088078,0.132999,0.132999,-0.012002,0.000022,0.000085
381,jul_aug,june_separate,coupon,6.5,2,2,150016,41.443507,False,False,False,0.129176,-0.040649,0.138265,-0.138265,-0.009090,0.000053,0.000178
396,jul_aug,june_separate,overall,all,2,2,498548,145.026046,False,False,False,0.123334,0.051292,0.097308,0.055481,0.026026,0.000031,0.000112
559,jul_sep,before,coupon,6.0,3,3,519789,154.344034,False,True,False,0.557572,0.557572,0.613198,0.613198,-0.055626,-0.000031,-0.000089


## Step 14 — Inspect small loans, delinquency and large-loan gains
`small_all_ages`: original balance <150k, all ages/statuses. `status`: all delinquent loans by coupon. `status_balance`: payment status × original-balance band, exposing small delinquent loans. `focus_current_young`: repeats prior current 12–23-month groups with incentive 0.5–1.0 for 6.0, and 1.0–1.5 for 6.5.

Review every month/stage. Existing small-group flags and event counts inform caution, not deletion. Absent groups are not zero-error groups. Do not hide worse groups in an improved overall average.


In [17]:
small=RESULTS.loc[RESULTS.grouping.eq('small_all_ages')]
delinquent=RESULTS.loc[RESULTS.grouping.eq('status')&RESULTS['group'].str.endswith('_delinquent')]
intersection=RESULTS.loc[RESULTS.grouping.eq('status_balance')&RESULTS['group'].str.contains('_delinquent_')]
focus=RESULTS.loc[RESULTS.grouping.eq('focus_current_young')]
for name,frame in [('monthly_small_loans',small),('monthly_delinquent_loans',delinquent),
                   ('monthly_delinquent_by_balance',intersection),('monthly_young_current_focus',focus)]:
    frame.to_csv(ROOT/(name+'.csv'),index=False)
    print(name);display(frame.loc[frame.stage.eq('before'),cols+['small_group_attention']])
priority=PERIODS.loc[PERIODS.grouping.eq('small_all_ages')|
                    (PERIODS.grouping.eq('status')&PERIODS['group'].str.endswith('_delinquent'))|
                    PERIODS.grouping.eq('focus_current_young')]
priority.to_csv(ROOT/'priority_group_period_summary.csv',index=False)
display(priority)
category=pd.concat([pd.read_csv(ROOT/'months'/mo/'category_coverage.csv') for mo in follow.MONTHS],ignore_index=True)
category.to_csv(ROOT/'category_coverage_all_months.csv',index=False)
display(category.loc[category.unknown_rows.gt(0)])


monthly_small_loans


,month,stage,group,loans,payoff_events,actual_payoff_pct,production_predicted_pct,expanded_predicted_pct,production_bias_pp,expanded_bias_pp,abs_bias_gain_pp,brier_gain,logloss_gain,outcome_status,calibration_outcome_status,small_group_attention
161,2026-07-01,before,6.0_lt150k,24369,303,1.194685,1.322614,1.427601,0.127930,0.232916,-0.104986,0.000002,0.000040,unconfirmed,not_applicable,False
170,2026-07-01,before,6.5_lt150k,15263,221,1.348038,1.589642,1.728488,0.241603,0.380450,-0.138846,0.000036,-0.000052,unconfirmed,not_applicable,False
690,2026-08-01,before,6.0_lt150k,24066,307,1.157280,1.295289,1.407212,0.138010,0.249932,-0.111922,0.000037,0.000186,unconfirmed,not_applicable,False
699,2026-08-01,before,6.5_lt150k,15042,209,1.313701,1.574152,1.714007,0.260451,0.400306,-0.139855,0.000058,-0.000098,unconfirmed,not_applicable,False
1203,2026-09-01,before,6.0_lt150k,23759,0,0.000000,1.141749,1.211865,1.141749,1.211865,-0.070117,-0.000108,-0.000766,provisional,not_applicable,True
1212,2026-09-01,before,6.5_lt150k,14833,0,0.000000,1.444111,1.544822,1.444111,1.544822,-0.100711,-0.000137,-0.001088,provisional,not_applicable,True


monthly_delinquent_loans


,month,stage,group,loans,payoff_events,actual_payoff_pct,production_predicted_pct,expanded_predicted_pct,production_bias_pp,expanded_bias_pp,abs_bias_gain_pp,brier_gain,logloss_gain,outcome_status,calibration_outcome_status,small_group_attention
15,2026-07-01,before,6.0_delinquent,3335,174,5.092926,3.892851,6.658173,-1.200075,1.565248,-0.365173,-0.000342,0.000592,unconfirmed,not_applicable,False
95,2026-07-01,before,6.5_delinquent,2156,123,5.682872,4.242315,6.523368,-1.440557,0.840496,0.600061,0.000631,0.003344,unconfirmed,not_applicable,False
548,2026-08-01,before,6.0_delinquent,3258,178,6.270465,4.050109,6.179346,-2.220357,-0.091119,2.129238,0.001526,0.006490,unconfirmed,not_applicable,False
624,2026-08-01,before,6.5_delinquent,2061,120,6.004506,4.261712,6.440071,-1.742794,0.435565,1.307229,0.001442,0.007101,unconfirmed,not_applicable,False
1061,2026-09-01,before,6.0_delinquent,3451,0,0.000000,3.538056,5.288070,3.538056,5.288070,-1.750013,-0.005332,-0.020843,provisional,not_applicable,True
1137,2026-09-01,before,6.5_delinquent,2150,0,0.000000,3.731475,5.716510,3.731475,5.716510,-1.985035,-0.005545,-0.023353,provisional,not_applicable,True


monthly_delinquent_by_balance


,month,stage,group,loans,payoff_events,actual_payoff_pct,production_predicted_pct,expanded_predicted_pct,production_bias_pp,expanded_bias_pp,abs_bias_gain_pp,brier_gain,logloss_gain,outcome_status,calibration_outcome_status,small_group_attention
163,2026-07-01,before,6.0_delinquent_lt150k,671,36,5.370236,3.928138,5.482582,-1.442099,0.112345,1.329754,0.000148,0.001410,unconfirmed,not_applicable,False
165,2026-07-01,before,6.0_delinquent_150_300k,1398,70,5.114410,3.779295,5.809251,-1.335115,0.694841,0.640275,-0.000850,-0.000504,unconfirmed,not_applicable,False
167,2026-07-01,before,6.0_delinquent_300_500k,817,51,6.338292,3.844778,6.983857,-2.493514,0.645565,1.847949,0.001758,0.008769,unconfirmed,not_applicable,False
169,2026-07-01,before,6.0_delinquent_500kplus,449,17,3.662375,4.055057,7.484025,0.392682,3.821650,-3.428968,-0.002190,-0.007260,unconfirmed,not_applicable,True
172,2026-07-01,before,6.5_delinquent_lt150k,467,28,5.616962,4.307537,6.376651,-1.309424,0.759689,0.549735,0.001601,0.003870,unconfirmed,not_applicable,True
174,2026-07-01,before,6.5_delinquent_150_300k,811,41,5.062751,4.272221,6.084857,-0.790530,1.022106,-0.231576,0.001229,0.003472,unconfirmed,not_applicable,False
176,2026-07-01,before,6.5_delinquent_300_500k,527,35,6.633840,3.947028,6.279578,-2.686812,-0.354263,2.332550,0.000883,0.006777,unconfirmed,not_applicable,False
178,2026-07-01,before,6.5_delinquent_500kplus,351,19,5.352581,4.463379,7.112163,-0.889203,1.759582,-0.870379,-0.000262,0.000131,unconfirmed,not_applicable,True
692,2026-08-01,before,6.0_delinquent_lt150k,651,25,3.739931,3.888917,4.962287,0.148986,1.222356,-1.073370,0.001284,0.002184,unconfirmed,not_applicable,True
694,2026-08-01,before,6.0_delinquent_150_300k,1387,67,4.999282,3.737231,5.388450,-1.262051,0.389169,0.872882,0.001791,0.005154,unconfirmed,not_applicable,False


monthly_young_current_focus


,month,stage,group,loans,payoff_events,actual_payoff_pct,production_predicted_pct,expanded_predicted_pct,production_bias_pp,expanded_bias_pp,abs_bias_gain_pp,brier_gain,logloss_gain,outcome_status,calibration_outcome_status,small_group_attention
2,2026-07-01,before,6.0_150_300k,8293,120,1.410697,1.345994,1.473085,-0.064702,0.062388,0.002314,1.313076e-07,0.000033,unconfirmed,not_applicable,False
3,2026-07-01,before,6.0_300_500k,6546,89,1.307617,1.516492,1.532509,0.208875,0.224892,-0.016017,-7.591969e-06,-0.000222,unconfirmed,not_applicable,False
4,2026-07-01,before,6.0_lt150k,2359,25,1.096683,1.313585,1.428843,0.216902,0.332160,-0.115257,2.204619e-06,0.000161,unconfirmed,not_applicable,True
5,2026-07-01,before,6.0_500kplus,2955,60,1.920134,1.943916,2.158764,0.023782,0.238631,-0.214849,-2.102222e-05,-0.000669,unconfirmed,not_applicable,False
82,2026-07-01,before,6.5_150_300k,2763,58,2.110139,1.984506,2.052531,-0.125633,-0.057607,0.068026,2.986820e-05,0.000840,unconfirmed,not_applicable,False
83,2026-07-01,before,6.5_300_500k,2000,38,1.824338,2.468438,2.373355,0.644100,0.549017,0.095083,-2.307455e-05,-0.000406,unconfirmed,not_applicable,False
84,2026-07-01,before,6.5_lt150k,1308,23,1.568895,1.572679,1.608853,0.003785,0.039959,-0.036174,2.749179e-05,0.000386,unconfirmed,not_applicable,True
85,2026-07-01,before,6.5_500kplus,938,23,2.468827,2.838057,2.642036,0.369231,0.173209,0.196021,-3.440388e-05,-0.000176,unconfirmed,not_applicable,True


,period,stage,grouping,group,months,expected_months,loan_months,upb_month_exposure_bn,all_outcomes_confirmed,calibration_outcomes_confirmed,ready_for_final_interpretation,production_weighted_abs_bias_pp,production_weighted_signed_bias_pp,expanded_weighted_abs_bias_pp,expanded_weighted_signed_bias_pp,weighted_abs_bias_gain_pp,weighted_brier_gain,weighted_logloss_gain
24,jul_aug,before,focus_current_young,6.0_150_300k,1,2,8293,1.839453,False,True,False,0.064702,-0.064702,0.062388,0.062388,0.002314,1.313076e-07,0.000033
25,jul_aug,before,focus_current_young,6.0_300_500k,1,2,6546,2.367550,False,True,False,0.208875,0.208875,0.224892,0.224892,-0.016017,-7.591969e-06,-0.000222
26,jul_aug,before,focus_current_young,6.0_500kplus,1,2,2955,1.787412,False,True,False,0.023782,0.023782,0.238631,0.238631,-0.214849,-2.102222e-05,-0.000669
27,jul_aug,before,focus_current_young,6.0_lt150k,1,2,2359,0.245220,False,True,False,0.216902,0.216902,0.332160,0.332160,-0.115257,2.204619e-06,0.000161
28,jul_aug,before,focus_current_young,6.5_150_300k,1,2,2763,0.605567,False,True,False,0.125633,-0.125633,0.057607,-0.057607,0.068026,2.986820e-05,0.000840
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
926,jul_sep,june_separate,focus_current_young,6.5_lt150k,1,3,1308,0.131396,False,False,False,0.202569,-0.202569,0.196636,-0.196636,0.005933,2.681668e-05,0.000400
944,jul_sep,june_separate,small_all_ages,6.0_lt150k,3,3,72194,7.447528,False,False,False,0.351784,0.300355,0.363556,0.363556,-0.011772,-3.191844e-06,0.000024
945,jul_sep,june_separate,small_all_ages,6.5_lt150k,3,3,45138,4.530964,False,False,False,0.441944,0.441944,0.527747,0.527747,-0.085803,1.327631e-05,-0.000134
1055,jul_sep,june_separate,status,6.0_delinquent,3,3,10044,2.855031,False,False,False,2.501310,-0.380748,2.105429,1.544003,0.395882,-5.097357e-04,-0.001515


,month,feature,unknown_rows,unknown_upb_share


## Step 15 — Interpret the evidence
1. Start with July/August **before**: do aggregate error, Brier and log loss improve together? Does mature September agree?
2. Check whether young-large gains remain without unacceptable small-loan/delinquent errors. Report magnitudes and events, not only win/loss labels.
3. Review June sensitivities separately. A sign change after separate June factors means calibration contributes. The common factor is diagnostic, not a deployment recommendation.
4. Rally improvement followed by later overprediction would support the concern of overreaction outside the rally. It would not justify deployment.
5. Broad stable later-calendar gains would support a subsequent paired-seed robustness experiment. This notebook never retrains.

This does not isolate splitting: agency mix, sampling, engine and stopping settings still differ. It does not test curtailment or served S-curves, and supplies no new Freddie result.

## Step 16 — Create and download checkpoints
Run after setup through Step 4, including partial progress after later failures. FULL contains selected original assets, exact code, specs, audits and completed monthly inputs/predictions. SMALL cannot restore model/data. Do not export inside ROOT. Original ZIPs are unchanged.

The panel expires in 30 days. Backups preserve downloaded months, not unscored expired tables. After Step 12, all four months' input/prediction files should be backed up.


In [18]:
from datetime import datetime,timezone
EXPORT_DIR=Path('/content/g1_julsep_downloads');EXPORT_DIR.mkdir(exist_ok=True)
assert not EXPORT_DIR.resolve().is_relative_to(ROOT.resolve())
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')+'_'+uuid.uuid4().hex[:8]
BACKUPS={}
for is_full in [False,True]:
    kind='FULL_recovery' if is_full else 'small_results'
    destination=EXPORT_DIR/(ROOT.name+'_'+stamp+'_'+kind+'.zip');hashes={}
    with zipfile.ZipFile(destination,'w',zipfile.ZIP_DEFLATED) as z:
        for p in ROOT.rglob('*'):
            if not p.is_file() or p.name.endswith('.recovering'):continue
            if not is_full and p.suffix in ['.parquet','.ubj','.joblib']:continue
            relative=p.relative_to(ROOT).as_posix();z.write(p,relative);hashes[relative]=core.sha(p)
        z.writestr('julsep_recovery_hashes.json',json.dumps(hashes,indent=2))
    BACKUPS[kind]=destination
    print(kind,':',destination,'bytes:',destination.stat().st_size,'SHA256:',core.sha(destination))
print('Download cells separately. Save BOTH ZIPs to Y:/OASIVE.')


small_results : /content/g1_julsep_downloads/g1_julsep_saved_models_20261101_20261102T002346_c40b5c5d_small_results.zip bytes: 902424 SHA256: baacfb118540b277b9fa3e35e696121bf1d963002c7f910a2a385ff44797149b
FULL_recovery : /content/g1_julsep_downloads/g1_julsep_saved_models_20261101_20261102T002346_c40b5c5d_FULL_recovery.zip bytes: 106074916 SHA256: eac17894f5e509f2914b8b0efd85e44956edba55b9c8d093f543a7c77abd515b
Download cells separately. Save BOTH ZIPs to Y:/OASIVE.


In [19]:
from google.colab import files
files.download(str(BACKUPS['small_results']))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
from google.colab import files
files.download(str(BACKUPS['FULL_recovery']))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Resume and troubleshooting
- Runtime loss: reopen this notebook, upload its own newest FULL checkpoint, set `RESTORE_FOLLOWUP_ZIP`, rerun setup and checked stages. Verified queries/months are reused. Keep ROOT/code unchanged.
- BadZipFile: wait for upload completion; select actual FULL ZIP, not shortcut or HTML page.
- Wrong hashes/model/replay: stop, do not retrain/replace assets to bypass checks.
- Changed source/panel/code/spec: preserve checkpoint; use a deliberate new run rather than rewriting identity files.
- Missing labels/features: inspect audit; do not impute missing payoff outcomes as zero or apply different filters by model. Curtailment eligibility alone is not a payoff-label error.
- Incomplete September: keep provisional and report July/August separately. A newer label vintage needs a separately named experiment, with both arms using identical revised labels and a reconciliation.
- Absent/small groups: disclose counts/events. Absence does not imply zero error.

## Outputs to share
`monthly_overall.csv`, `monthly_coupon.csv`, `period_summary.csv`, `june_calibration_sensitivity.csv`, `june_factors.json`, `june_outcome_review.json`, `monthly_small_loans.csv`, `monthly_delinquent_loans.csv`, `monthly_delinquent_by_balance.csv`, `monthly_young_current_focus.csv`, `priority_group_period_summary.csv`, `outcome_maturity_review.csv`, `monthly_outcome_coverage.csv`, `category_coverage_all_months.csv`, `verified_inventory_8_cells.csv`, `source_metadata.csv`, `completion.json`.

Keep **before** primary. June never enters the evaluation summary. All June factors are fixed across later months and subgroups.

Technical reference: [BigQuery ML.PREDICT](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-predict). No CREATE MODEL or trainer job is executed.
